In [ ]:
import numpy as  np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
titanic_data = pd.read_csv("train.csv")

In [ ]:
titanic_data.head()

In [ ]:
titanic_data.describe()

In [ ]:
titanic_data.head()

In [ ]:
sns.heatmap(titanic_data.select_dtypes(include = np.number).corr(),cmap = "YlGnBu")

In [ ]:
ax = titanic_data["Age"].hist(bins=15, density=True, stacked=True, color='teal', alpha=0.6)
titanic_data["Age"].plot(kind='density', color='teal')
ax.set(xlabel='Age')
plt.xlim(-10,85)
plt.show()

In [ ]:
titanic_data["Age"].mean(skipna=True)

In [ ]:
titanic_data["Age"].median(skipna=True)

In [ ]:
sns.countplot(x='Embarked', hue='Embarked', data=titanic_data, palette='Set2', legend=False)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
split = StratifiedShuffleSplit(n_splits=1,test_size = 0.2, random_state = 42)

In [ ]:
for train_indices, test_indices in split.split(titanic_data,titanic_data[["Survived", "Pclass", "Sex"]]):
    strat_train_set = titanic_data.loc[train_indices]
    strat_test_set = titanic_data.loc[test_indices]

In [ ]:
plt.subplot(1,2,1)
strat_train_set['Survived'].hist()
strat_train_set['Pclass'].hist()

plt.subplot(1,2,2)
strat_test_set['Survived'].hist()
strat_test_set['Pclass'].hist()

In [ ]:
strat_train_set.info()

In [ ]:
strat_train_set.isnull().sum()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

class AgeImputer(BaseEstimator , TransformerMixin) :
    def fit(self,X,y=None):
        return self

    def transform(self , X):
        imputer = SimpleImputer(strategy = "median")
        X['Age'] =  imputer.fit_transform(X[['Age']])
        return X


In [ ]:
from sklearn.preprocessing import OneHotEncoder

class FeatureEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        encoder = OneHotEncoder()
        matrix = encoder.fit_transform(X[['Embarked']]).toarray()
        column_names = ["C", "S", "Q", "N"]
        for i in range(len(matrix.T)):
            X[column_names[i]] = matrix.T[i]
        
        matrix = encoder.fit_transform(X[['Sex']]).toarray()
        column_names = ["Female", "Male"]
        for i in range(len(matrix.T)):
            X[column_names[i]] = matrix.T[i]
        
        return X


In [ ]:
class FeatureDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(["Embarked","Name","Ticket","Cabin","Sex","N"],axis =1, errors = "ignore")


In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([("ageimputer", AgeImputer()),
                     ("featureencoder", FeatureEncoder()),
                     ("featuredropper", FeatureDropper())])


In [ ]:
strat_train_set = pipeline.fit_transform(strat_train_set)

In [ ]:
strat_train_set

In [ ]:
strat_train_set.info()

In [ ]:
strat_train_set.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X = strat_train_set.drop(["Survived"], axis=1)
y = strat_train_set["Survived"]
X_data = scaler.fit_transform(X)
y_data = y.to_numpy()

In [ ]:
X_data

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
clf = RandomForestClassifier(random_state =42)

param_gird = [
    {"n_estimators": [500],  "max_depth":[5 ,10], "min_samples_split":[2,4,6]}
]

grid_search = GridSearchCV( clf , param_gird, cv=5 , scoring = "accuracy", return_train_score = True )
grid_search.fit(X_data, y_data)

In [ ]:
final_clf = grid_search.best_estimator_

In [ ]:
final_clf

In [ ]:
strat_test_set = pipeline.fit_transform(strat_test_set)

In [ ]:
X_test = strat_test_set.drop(["Survived"], axis=1)
y_test = strat_test_set["Survived"]
scalar = StandardScaler()
X_data_test = scaler.fit_transform(X_test)
y_data_test = y_test.to_numpy()

In [ ]:
final_clf.score(X_data_test,y_data_test)

In [ ]:
titanic_data

In [ ]:
final_data = pipeline.fit_transform(titanic_data)

In [ ]:
final_data

In [ ]:
X_final = final_data.drop(["Survived"],axis = 1 )
y_final = final_data["Survived"]

scaler = StandardScaler()

X_data_final = scaler.fit_transform(X_final)
y_data_final = y_final.to_numpy()

In [ ]:
prod_clf = RandomForestClassifier(random_state = 42)

param_gird = [
    {"n_estimators": [500],  "max_depth":[5 ,10], "min_samples_split":[2,4,6]}
]

grid_search = GridSearchCV( prod_clf , param_gird, cv=5 , scoring = "accuracy", return_train_score = True )
grid_search.fit(X_data_final, y_data_final)

In [ ]:
prod_final_clf =  grid_search.best_estimator_

In [ ]:
prod_final_clf

In [ ]:
titanic_test_data = pd.read_csv("test.csv")

In [ ]:
titanic_test_data

In [ ]:
final_test_data = pipeline.fit_transform(titanic_test_data)

In [ ]:
final_test_data

In [ ]:
final_test_data.info()

In [ ]:
X_final_test = final_test_data
X_final_test = X_final_test.ffill()

scaler = StandardScaler()
X_data_final_test = scaler.fit_transform(X_final_test)


In [ ]:
predictions = prod_final_clf.predict(X_data_final_test)

In [ ]:
predictions

In [ ]:
final_df = pd.DataFrame(titanic_test_data["PassengerId"])


In [ ]:
final_df["Survived"] = predictions

In [ ]:
final_df.to_csv("pedictions2.csv" ,index=False)

In [ ]:
final_df